# Exploratory Data Analysis (EDA): Music Context & Structure

**Course**: CSE425 / EEE474 / CSE715 Neural Networks Project  
**Focus**: Audio features, tag distributions, and graph topology for music understanding.

This notebook covers:
1. **Audio Feature Exploration**: Log-mel spectrograms, chromagrams, and MFCC representations.
2. **Music Tag Taxonomy**: Multi-label distribution and co-occurrence patterns.
3. **Graph Topology Analysis**: Segment similarity networks and chord progression transitions.

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import yaml

from src.audio_features import AudioFeatureExtractor, generate_synthetic_audio, set_seed, load_config
from src.graph_builder import build_segment_graph

config = load_config('config.yaml')
set_seed(config.get('seed', 42))
print('Config loaded successfully.')

## 1. Feature Representation Analysis
Inspect the frequency and harmonic characteristics of music audio windows.

In [ ]:
extractor = AudioFeatureExtractor.from_config('config.yaml')
audio = generate_synthetic_audio(duration=24.0, sample_rate=extractor.sample_rate)

mel = extractor.compute_log_mel_spectrogram(audio)
chroma = extractor.compute_chroma(audio)
mfcc = extractor.compute_mfcc(audio)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].imshow(mel, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Normalized Log-Mel Spectrogram (128 bins)')
axes[0].set_ylabel('Mel Bin')

axes[1].imshow(chroma, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('12-Class Harmonic Chroma Features')
axes[1].set_ylabel('Pitch Class')

axes[2].imshow(mfcc, aspect='auto', origin='lower', cmap='coolwarm')
axes[2].set_title('Mel-Frequency Cepstral Coefficients (20 MFCCs)')
axes[2].set_ylabel('MFCC Bin')
axes[2].set_xlabel('Time Frames')

plt.tight_layout()
plt.show()

## 2. Music Segment Graph Connectivity
Analyze edge density, degree distributions, and similarity graph topology.

In [ ]:
node_features = extractor.extract_segment_features(audio)
graph = build_segment_graph(node_features, similarity_threshold=0.7, add_temporal_edges=True)

print(f'Graph Nodes (Segments): {graph.num_nodes}')
print(f'Graph Edges (Connections): {graph.num_edges}')
print('Edge Index: \n', graph.edge_index.numpy())
print('Edge Weights: \n', graph.edge_attr.numpy().flatten())